In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ \text{ploss} $$

$$ 
\mathbb{R} \times \mathbb{R} \to \mathbb{R}, \quad y(p, t) = 
\begin{cases} 0 & \text{if } pt > 0 \\ -pt & \text{if } pt \leq 0 \end{cases} 
$$

$$ 
\mathbb{R^n \times \mathbb{R^n}} \to \mathbb{R^n}, \quad y(\mathbf{p}, \mathbf{t}) = 
(y(p_1, t_1), y(p_2, t_2), \dots, y(p_n, t_n)) 
$$

$$ 
\frac{\partial y}{\partial p_i} = 
\begin{cases} 
    0 & \text{if } p_i t_i > 0 \\ 
    -t_i & \text{if } p_i t_i \leq 0 
\end{cases} 
$$

$$ d\mathbf{y} = J_y(\mathbf{p}) \, d\mathbf{p} $$

$$ \text{Frobenius equality} $$

$$
\left \langle \frac{dF}{dp}, d\mathbf{p} \right \rangle =
\left \langle \frac{dF}{dy}, d\mathbf{y} \right \rangle
$$

$$ \\[2em] $$

$$
\left \langle \frac{dF}{dy}, d\mathbf{y} \right \rangle =
\left \langle \frac{dF}{dy}, J_y(\mathbf{p}) \, d\mathbf{p} \right \rangle =
\left \langle J_y(\mathbf{p}) ^\top \, \frac{dF}{dy}, d\mathbf{p} \right \rangle \implies
$$

$$ 
\frac{dF}{dp} = 
J_y(\mathbf{p})^\top \, \frac{dF}{dy} = 
\frac{dy}{dp} \odot \frac{dF}{dy} 
$$

In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
from approx import approx # type: ignore

def ploss(p, t, reduction = "mean"):
    """Calculate the `perceptron loss` between predicted `p` logits and true targets `t`."""

    assert tr.all((t == -1.0) | (t == +1.0))

    mask = (p * t <= 0)
    loss = tr.zeros_like(p)
    loss[mask] = - p[mask] * t[mask]
    
    if reduction != "mean":
        assert False, f"Unsupported reduction: {reduction}"

    return loss.mean()


class PLossFunction(autograd.Function):
    """Custom autograd function for the `perceptron loss`."""

    @staticmethod
    def forward(ctx, p, t):
        ctx.save_for_backward(p, t)
        return ploss(p, t)

    @staticmethod
    def backward(ctx, dF_dy):
        (p, t) = ctx.saved_tensors
        S = p.shape[0]
        FO = p.shape[1]
        N = S * FO

        mask = (p * t <= 0)
        dy_dz = tr.zeros_like(p)
        dy_dz[mask] = -t[mask]
        dy_dz = 1 / N * dy_dz

        dF_dz = dy_dz * dF_dy
        
        return (dF_dz, None)
    

class PLoss(nn.Module):
    """Custom module for the `perceptron loss`."""

    def __init__(self, reduction: str = "mean"):
        super().__init__()

        if reduction != "mean":
            assert False, f"Unsupported reduction: {reduction}"
    
    def forward(self, z: tr.Tensor, t: tr.Tensor) -> tr.Tensor:
        return PLossFunction.apply(z, t)

In [ ]:
# Unit tests

def test_gradcheck(samples=10):
    def fn(p, t):
        return PLossFunction.apply(p, t)

    p = tr.randn(samples, 1, dtype=tr.float64, requires_grad=True)

    t = tr.where(
        tr.rand(samples, 1) < 0.5,
        tr.tensor(-1.0, dtype=tr.float64),
        tr.tensor(+1.0, dtype=tr.float64)
    )

    assert autograd.gradcheck(fn, (p, t), eps=1e-6, atol=1e-4, rtol=1e-4)


def test_compare(samples):
    p = tr.randn(samples, 1, requires_grad=True)

    t = tr.where(
        tr.rand(samples, 1) < 0.5,
        tr.tensor(-1.0),
        tr.tensor(+1.0)
    ).reshape(samples, 1)

    p1 = p.clone().detach().requires_grad_(True)
    y1 = PLoss()(p1, t)
    y1.backward()

    p2 = p.clone().detach().requires_grad_(True)
    y2 = tr.max(tr.zeros_like(p2), - p2 * t).mean()
    y2.backward()

    assert y1.item() == approx(y2.item())
    assert p1.grad == approx(p2.grad)


if __name__ == "__main__":
    test_gradcheck(1)
    test_gradcheck(10)
    test_gradcheck(100)

    test_compare(1)
    test_compare(10)
    test_compare(100)